# 혈당코치 우땅이: 검색 평가 (BM25 · Reranker 비교)

스터디 3주차 과제(**Reranking 적용 후 성능 비교**)를 위한 노트북이에요. 지금 검색(FAISS + 규칙 가감점)과 몇 가지 개선안을 **같은 질문**으로 돌려서 숫자로 비교해요.

| 이름 | 방식 |
|---|---|
| A 현재 | FAISS 검색 + 상황·대상 가감점 (지금 챗봇이 쓰는 `guideline_rag.py` 그대로) |
| B Dense만 | A에서 가감점을 뺀 순수 임베딩 검색 |
| C BM25만 | 단어 기반(Sparse) 검색 |
| D Dense+BM25 | B와 C 결과를 RRF로 합친 하이브리드 |
| E D+Reranker | D의 상위 20개를 Cross-encoder 리랭커로 다시 정렬 |
| F A+Reranker | A의 상위 20개를 리랭커로 다시 정렬 (지금 챗봇에 리랭커만 붙이는 경우) |

**지표**: `Recall@1`(1위가 정답 문서), `Recall@4`(상위 4개 안에 정답, 챗봇이 실제로 쓰는 개수), `MRR@10`(정답이 위에 있을수록 높음)

**평가 질문**: 문서 조각을 무작위로 골라서 LLM이 "이 조각이 답이 되는 질문"을 만들어요 (정답 = 그 조각). `평가/eval_questions.json`에 저장되니 다음부터는 같은 질문으로 비교해요. 직접 만든 질문도 아래 `MANUAL_QA`에 넣을 수 있어요.

> ⚠️ 자동 질문은 문서 내용에서 만들어서 실제 사용자 질문보다 쉬울 수 있어요. 숫자는 **방식끼리의 차이**를 볼 때 의미가 커요. 가능하면 `MANUAL_QA`에 직접 쓴 질문도 10개 정도 넣으세요.

**돌리는 순서**: 0 → 1 → 2 → 3(질문) → 4(검색 방식) → 5(비교) → 6(임계값) → (7 실패 사례) → 8(저장)

**먼저 확인**: `혈당코치/태영_코드/guideline_tools.py`, `guideline_rag.py`, `기능2/관련 문서/kdca/*.html`, Colab 🔑 Secrets의 `OPENAI_API_KEY`(노트북 접근 켜기). 리랭커는 GPU(런타임 → 런타임 유형 변경 → T4)가 있으면 빨라요.


## 0. 설치

In [ ]:
!pip -q install -U langchain langchain-openai langchain-community faiss-cpu sentence-transformers

## 1. 설정

In [ ]:
import os, sys, re, json, time, random, importlib, warnings
from collections import defaultdict, Counter
import numpy as np, pandas as pd
from google.colab import drive, userdata
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings

warnings.filterwarnings('ignore', message='.*langchain-community.*')
drive.mount('/content/drive')

BASE      = '/content/drive/MyDrive/혈당코치'
TOOL_DIRS = [f'{BASE}/민석_코드', f'{BASE}/태영_코드']
DOCS_DIR  = f'{BASE}/기능2/관련 문서'
EVAL_DIR  = f'{BASE}/평가'                      # 질문·결과를 저장할 폴더
os.makedirs(EVAL_DIR, exist_ok=True)

OPENAI_KEY  = userdata.get('OPENAI_API_KEY')
CHAT_MODEL  = 'gpt-4o-mini'
EMBED_MODEL = 'text-embedding-3-small'
client      = OpenAI(api_key=OPENAI_KEY)
embeddings  = OpenAIEmbeddings(model=EMBED_MODEL, api_key=OPENAI_KEY)

N_QUESTIONS  = 60       # 자동으로 만들 평가 질문 수
SEED         = 42
USE_RERANKER = True     # 리랭커를 안 쓰려면 False (E, F 열이 빠져요)
RERANK_MODEL = 'BAAI/bge-reranker-v2-m3'   # 다국어 Cross-encoder (처음 한 번 2GB 정도 받아요)
POOL         = 20       # 리랭커가 다시 줄 세울 후보 수

## 2. 검색 모듈 불러오기
챗봇이 쓰는 `GuidelineRAG`를 그대로 불러와요. 문서 청크 목록(`DOCS`)을 번호로 다뤄요.

In [ ]:
for d in TOOL_DIRS:
    if d not in sys.path:
        sys.path.append(d)
import guideline_tools as gt
import guideline_rag as grag
importlib.reload(gt); importlib.reload(grag)

rag  = grag.GuidelineRAG(DOCS_DIR, embeddings).build()
DOCS = rag.docs                                   # LangChain Document 목록 (번호 = 청크 번호)
print('청크 수:', len(DOCS))

def _key(m):
    return (m['doc_title'], m['section'], m['text'])

KEY2IDXS = {}                                     # (문서, 섹션, 본문) → 청크 번호들 (같은 글이 겹쳐도 처리)
HEAD2IDXS = {}                                    # (문서, 섹션, 본문 앞 80자) → 청크 번호들 (저장·불러오기용)
for i, d in enumerate(DOCS):
    m = d.metadata
    KEY2IDXS.setdefault(_key(m), []).append(i)
    HEAD2IDXS.setdefault((m['doc_title'], m['section'], m['text'][:80]), []).append(i)

dup = sum(len(v) > 1 for v in KEY2IDXS.values())
print('본문이 똑같은 청크(중복):', dup, '건 /', '평균 청크 길이', round(sum(len(d.metadata['text']) for d in DOCS) / len(DOCS)), '자')

def gold_key(i):
    m = DOCS[i].metadata
    return [m['doc_title'], m['section'], m['text'][:80]]

def resolve(keys):
    """저장해 둔 정답 키 → 현재 청크 번호 집합. 문서가 바뀌어 못 찾으면 빈 집합"""
    out = set()
    for k in keys:
        out |= set(HEAD2IDXS.get(tuple(k), []))
    return out

## 3. 평가 질문 만들기
- 문서 조각 `N_QUESTIONS`개를 무작위로 골라, LLM이 일상어로 질문을 하나씩 만들어요. 문서 속 단어를 그대로 쓰지 않게 해서 BM25에 유리해지는 걸 줄였어요.
- 대상이 임산부·체중조절인 조각은 질문에 그 상황이 들어가게 해요.
- 한 번 만들면 `평가/eval_questions.json`에 저장돼요. 질문을 바꾸고 싶으면 그 파일을 지우고 다시 돌리세요. (직접 고쳐도 돼요)
- `MANUAL_QA`: 직접 쓴 질문. `doc`은 문서 이름 일부, `keyword`는 정답 청크 본문에 들어 있는 단어예요.

In [ ]:
QFILE = f'{EVAL_DIR}/eval_questions.json'

QGEN_PROMPT = '''아래는 당뇨 식사요법 안내 문서의 일부예요. 이 글을 읽고 답을 찾을 수 있는 질문을 한국어로 하나 만들어 주세요.

규칙:
- 일반 사용자가 챗봇에 물을 법한 자연스러운 말투로 쓰세요. (예: "외식할 때 뭐 조심해야 해?")
- 문서에 나온 어려운 단어나 문장을 그대로 베끼지 말고, 일상어로 바꿔 쓰세요.
- 이 글이 어떤 문서의 어느 부분인지 몰라도 이해되는 질문이어야 해요. ("이 문서", "위 내용" 같은 말 금지)
- 대상이 "임산부"이면 질문에 임신 중이라는 상황을, "체중조절"이면 체중 감량 목적을 자연스럽게 넣으세요. "일반"이면 넣지 마세요.

문서: {doc_title}
소제목: {section}
대상: {target}
내용:
{text}

JSON으로만 답하세요: {{"question": "..."}}'''

# 직접 만든 질문 (선택). 예시:
# MANUAL_QA = [{'q': '술 마실 때 안주는 뭐가 좋아?', 'doc': '당뇨환자의 식이요법', 'keyword': '음주', 'target': '일반'}]
MANUAL_QA = []

def make_questions(n, seed):
    rnd = random.Random(seed)
    pool = [i for i, d in enumerate(DOCS) if len(d.metadata['text']) >= 120]
    out = []
    for i in rnd.sample(pool, min(n, len(pool))):
        m = DOCS[i].metadata
        resp = client.chat.completions.create(
            model=CHAT_MODEL, temperature=0.7, response_format={'type': 'json_object'},
            messages=[{'role': 'user', 'content': QGEN_PROMPT.format(
                doc_title=m['doc_title'], section=m['section'], target=m['target'], text=m['text'][:1200])}])
        q = str(json.loads(resp.choices[0].message.content).get('question', '')).strip()
        if q:
            out.append({'q': q, 'target': m['target'], 'source': '자동', 'gold_keys': [gold_key(i)]})
    return out

def manual_items():
    out = []
    for it in MANUAL_QA:
        idxs = [i for i, d in enumerate(DOCS)
                if it['doc'] in d.metadata['doc_title'] and it['keyword'] in d.metadata['text']]
        if not idxs:
            print('⚠️ 정답 청크를 못 찾았어요:', it['q']); continue
        out.append({'q': it['q'], 'target': it.get('target', '일반'), 'source': '직접',
                    'gold_keys': [gold_key(i) for i in idxs]})
    return out

if os.path.exists(QFILE):
    QS = json.load(open(QFILE, encoding='utf-8'))
    print('저장된 질문 불러옴:', len(QS))
else:
    QS = make_questions(N_QUESTIONS, SEED)
    json.dump(QS, open(QFILE, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    print('질문 새로 만듦:', len(QS))

QS = [q for q in QS if q.get('source') != '직접'] + manual_items()
for q in QS:
    q['gold'] = resolve(q['gold_keys'])
QS = [q for q in QS if q['gold']]                      # 문서가 바뀌어 정답을 못 찾는 질문은 뺌
print('평가 질문:', len(QS), '개 (자동', sum(q['source'] == '자동' for q in QS), '/ 직접', sum(q['source'] == '직접' for q in QS), ')')
pd.DataFrame([{'질문': q['q'], '대상': q['target'], '정답 문서': DOCS[min(q['gold'])].metadata['doc_title']} for q in QS]).head(10)

## 4. 검색 방식 만들기
모든 방식이 "청크 번호 목록(위가 1위)"을 돌려줘요.
- **BM25**는 한국어 형태소 분석기 없이 **글자 2개 단위**로 잘라서 계산해요. (조사가 붙어도 잘 맞는 편이에요)
- **RRF**: 두 검색의 순위만 보고 합쳐요 (점수 단위가 달라도 돼요).
- **리랭커**: 질문과 문서 조각을 함께 읽고 관련도를 다시 계산하는 Cross-encoder예요.

In [ ]:
def _dedupe(seq):
    seen, out = set(), []
    for i in seq:
        if i not in seen:
            seen.add(i); out.append(i)
    return out

def _first_idx(meta_or_hit):
    return KEY2IDXS[_key(meta_or_hit)][0]

# ── B. Dense만 (가감점 없음) ──
def dense_rank(q):
    pairs = rag.vs.similarity_search_with_score(q, k=len(DOCS))
    return _dedupe([_first_idx(d.metadata) for d, _ in pairs])        # 점수 높은 순으로 와요

# ── C. BM25 (글자 2개 단위) ──
def _tok(s):
    s = re.sub(r'[^0-9A-Za-z가-힣]', '', s)
    return [s[i:i + 2] for i in range(len(s) - 1)] or [s]

class BM25:
    def __init__(self, texts, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.tf = [Counter(_tok(t)) for t in texts]
        self.len = np.array([sum(c.values()) for c in self.tf], dtype=float)
        self.avg = self.len.mean()
        df = Counter()
        for c in self.tf:
            df.update(c.keys())
        n = len(texts)
        self.idf = {w: np.log(1 + (n - f + 0.5) / (f + 0.5)) for w, f in df.items()}
    def rank(self, q):
        scores = np.zeros(len(self.tf))
        for w in set(_tok(q)):
            if w not in self.idf:
                continue
            for i, c in enumerate(self.tf):
                f = c.get(w, 0)
                if f:
                    scores[i] += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.len[i] / self.avg))
        return [int(i) for i in np.argsort(-scores, kind='stable')]

bm25 = BM25([d.page_content for d in DOCS])
def bm25_rank(q):
    return bm25.rank(q)

# ── D. 하이브리드 (RRF) ──
def rrf(*rank_lists, k=60):
    sc = defaultdict(float)
    for r in rank_lists:
        for pos, i in enumerate(r, 1):
            sc[i] += 1.0 / (k + pos)
    return [i for i, _ in sorted(sc.items(), key=lambda x: -x[1])]

def hybrid_rank(q):
    return rrf(dense_rank(q), bm25_rank(q))

# ── A. 현재 챗봇 검색 (rag.search 그대로, 거절 기준만 잠시 끔) ──
def current_rank(q, target='일반', k=POOL):
    situations = gt.tag_situations(q) if hasattr(gt, 'tag_situations') else []
    old, rag.min_score = rag.min_score, 0.0
    try:
        hits = rag.search(q, situations, target, k=k).get('결과', [])
    finally:
        rag.min_score = old
    idxs = [KEY2IDXS[(h['문서'], h['섹션'], h['내용'])][0] for h in hits]
    return _dedupe(idxs), (hits[0]['score'] if hits else 0.0)

# ── 리랭커 ──
reranker = None
if USE_RERANKER:
    import torch
    from sentence_transformers import CrossEncoder
    reranker = CrossEncoder(RERANK_MODEL, max_length=512, device='cuda' if torch.cuda.is_available() else 'cpu')
    print('리랭커 준비:', RERANK_MODEL, '/', 'GPU' if torch.cuda.is_available() else 'CPU (느려요)')

def rerank(q, ranked, pool=POOL):
    head, tail = ranked[:pool], ranked[pool:]
    scores = reranker.predict([(q, DOCS[i].page_content) for i in head], batch_size=16, show_progress_bar=False)
    order = [i for _, i in sorted(zip(scores, head), key=lambda x: -x[0])]
    return order + tail

VARIANTS = {
    'A 현재(FAISS+가감점)': lambda q, t: current_rank(q, t)[0],
    'B Dense만':           lambda q, t: dense_rank(q),
    'C BM25만':            lambda q, t: bm25_rank(q),
    'D Dense+BM25(RRF)':   lambda q, t: hybrid_rank(q),
}
if reranker is not None:
    VARIANTS['E D+Reranker'] = lambda q, t: rerank(q, hybrid_rank(q))
    VARIANTS['F A+Reranker'] = lambda q, t: rerank(q, current_rank(q, t)[0])
print('비교할 방식:', list(VARIANTS))

## 5. 비교 실행
질문마다 모든 방식을 돌리고, 정답이 몇 위에 나왔는지 기록해요. 질문 60개 기준으로 리랭커 포함 몇 분 걸려요.

정답을 세 가지 엄격함으로 세요. (문서에 비슷한 내용이 여러 곳에 나오면 "정답 청크 하나"만 정답으로 치면 점수가 너무 낮게 나와요)
- **청크**: 질문을 만든 바로 그 청크 (가장 엄격)
- **섹션**: 같은 문서의 같은 소제목에 속한 청크 (이웃 청크, 겹치는 내용 허용)
- **문서**: 같은 문서의 청크 (가장 느슨)

> 질문이 60개면 질문 1개 = 1.7%p예요. 방식끼리 1~3개 차이는 우연일 수 있어요. 차이가 의미 있는지는 아래 7번에서 질문을 직접 보고 판단하세요.

In [ ]:
def _sec(i):
    m = DOCS[i].metadata
    return (m['doc_title'], m['section'])

def _first_rank(ranked, ok):
    return next((p for p, i in enumerate(ranked, 1) if ok(i)), None)

def evaluate(qs):
    rows = []
    for n, item in enumerate(qs, 1):
        gold_secs = {_sec(i) for i in item['gold']}
        gold_docs = {s[0] for s in gold_secs}
        for name, fn in VARIANTS.items():
            ranked = fn(item['q'], item['target'])
            rows.append({'qid': n, '질문': item['q'], '출처': item['source'], '방식': name,
                         '순위': _first_rank(ranked, lambda i: i in item['gold']),               # 정답 청크
                         '섹션순위': _first_rank(ranked, lambda i: _sec(i) in gold_secs),         # 같은 문서·소제목의 청크
                         '문서순위': _first_rank(ranked, lambda i: DOCS[i].metadata['doc_title'] in gold_docs),
                         'top4': ranked[:4]})
        if n % 10 == 0:
            print(f'  {n}/{len(qs)}')
    return pd.DataFrame(rows)

def summarize(detail):
    def stat(g):
        r, s, d = g['순위'], g['섹션순위'], g['문서순위']
        return pd.Series({
            '청크 Recall@1': (r == 1).mean(),
            '청크 Recall@4': (r <= 4).mean(),
            '청크 Recall@10': (r <= 10).mean(),
            '청크 MRR@10': r.map(lambda x: 1 / x if pd.notna(x) and x <= 10 else 0).mean(),
            '섹션 Recall@4': (s <= 4).mean(),
            '문서 Recall@4': (d <= 4).mean(),
        })
    return detail.groupby('방식', sort=False).apply(stat).round(3)

DETAIL = evaluate(QS)
SUMMARY = summarize(DETAIL)
SUMMARY

In [ ]:
# 직접 만든 질문만 따로 보기 (있을 때)
if (DETAIL['출처'] == '직접').any():
    display(summarize(DETAIL[DETAIL['출처'] == '직접']))
else:
    print('MANUAL_QA가 비어 있어요. 직접 쓴 질문을 넣으면 여기서 따로 비교해요.')

## 6. 거절 기준(`min_score` 0.30) 점검
관련 없는 질문의 1위 점수와, 관련 있는 질문의 1위 점수를 비교해서 **0.30이 적당한지** 봐요.
- 관련 없는 질문이 0.30을 넘으면 → 엉뚱한 문서가 근거로 나가요 (거절 실패).
- 관련 있는 질문이 기준보다 낮으면 → 답할 수 있는 걸 거절해요 (오거절).

In [ ]:
OOD = ['오늘 서울 날씨 어때?', '아이폰 추천해줘', '파이썬에서 리스트 정렬하는 법 알려줘', '서울에서 부산 가는 기차 시간',
       '삼각함수 공식 알려줘', '주식 투자 어떻게 시작해?', '고양이 사료 추천해줘', '요즘 재밌는 영화 뭐 있어?',
       '환율 얼마야?', '축구 월드컵 우승국이 어디야?']

def top_score(q, target='일반'):
    return current_rank(q, target, k=1)[1]

ood_s = np.array([top_score(q) for q in OOD])
in_s  = np.array([top_score(q['q'], q['target']) for q in QS])
print(f'관련 있는 질문 1위 점수: 최소 {in_s.min():.3f} / 하위 5% {np.percentile(in_s, 5):.3f} / 중앙값 {np.median(in_s):.3f}')
print(f'관련 없는 질문 1위 점수: 중앙값 {np.median(ood_s):.3f} / 최대 {ood_s.max():.3f}')
rows = [{'임계값': t, '관련 없는 질문 거절률': (ood_s < t).mean(), '관련 있는 질문 오거절률': (in_s < t).mean()}
        for t in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55]]
display(pd.DataFrame(rows).round(3))
print('높은 점수 순 관련 없는 질문:'); 
for q, s in sorted(zip(OOD, ood_s), key=lambda x: -x[1])[:5]:
    print(f'  {s:.3f}  {q}')

## 7. (선택) 실패 사례 보기
정답이 4위 안에 못 든 질문을 방식별로 봐요. 질문이 애매하거나 정답이 여러 조각에 나뉘어 있어서일 수도 있어요.

In [ ]:
def show_fail(variant, n=8):
    d = DETAIL[(DETAIL['방식'] == variant) & ~(DETAIL['순위'] <= 4)]
    print(f'{variant}: 4위 안에 못 든 질문 {len(d)}개 / 전체 {len(QS)}개')
    for _, r in d.head(n).iterrows():
        print(' -', r['질문'], '→ 순위', r['순위'])

def better_than(a, b, n=8):
    """a 방식은 4위 안인데 b 방식은 아닌 질문"""
    da = DETAIL[DETAIL['방식'] == a].set_index('qid')
    db = DETAIL[DETAIL['방식'] == b].set_index('qid')
    hit = [i for i in da.index if (da.at[i, '순위'] <= 4) and not (db.at[i, '순위'] <= 4)]
    print(f'{a}는 되고 {b}는 안 된 질문: {len(hit)}개')
    for i in hit[:n]:
        print(' -', da.at[i, '질문'], f"({a} {da.at[i, '순위']:.0f}위 / {b} {db.at[i, '순위']})")

show_fail('A 현재(FAISS+가감점)')
if reranker is not None:
    better_than('E D+Reranker', 'A 현재(FAISS+가감점)')

### 7-2. 실패 질문 하나를 자세히 보기
질문 일부를 넣으면 **정답 청크**와 **그 방식이 가져온 상위 4개**를 나란히 보여줘요. 상위 4개가 정답은 아니어도 질문에 답이 되는 내용이면, 검색이 틀린 게 아니라 평가가 박한 거예요.

In [ ]:
def inspect_q(part, variant='A 현재(FAISS+가감점)', n=4):
    item = next((q for q in QS if part in q['q']), None)
    if item is None:
        print('질문을 못 찾았어요:', part); return
    print('질문:', item['q'], f"(대상 {item['target']})")
    print('── 정답 청크 ──')
    for i in sorted(item['gold'])[:2]:
        m = DOCS[i].metadata
        print(f"  [{m['doc_title']} > {m['section']}] {m['text'][:140]}...")
    ranked = VARIANTS[variant](item['q'], item['target'])
    print(f'── {variant} 상위 {n}개 ──')
    for p, i in enumerate(ranked[:n], 1):
        m = DOCS[i].metadata
        mark = '✅정답' if i in item['gold'] else ''
        print(f"  {p}위 [{m['doc_title']} > {m['section']}] {m['text'][:140]}... {mark}")

inspect_q('정기적으로 어떤 검사')
inspect_q('임신 중에 건강한 식사')

### 7-3. LLM이 판정하는 평가 (정답 청크 하나만 정답으로 치지 않는 방식)
"정답 청크"가 아니어도 **질문에 답이 될 내용**이 있으면 맞는 걸로 세요. 방식마다 상위 4개를 gpt-4o-mini에게 보여주고 "이 조각이 질문에 답이 되나?"를 판정하게 해요. (RAGAS의 context relevance와 같은 생각이에요)
- `LLM판정 Hit@4`: 상위 4개 중 답이 될 조각이 하나라도 있는 질문의 비율
- `Precision@4`: 상위 4개 중 답이 되는 조각의 비율
- 결과는 `평가/judge_cache.json`에 저장돼서 다시 돌려도 비용이 안 들어요. (질문·상위 4개가 같을 때)
> 판정도 LLM이라 완벽하지 않아요. 방식끼리 비교할 때 보세요.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

JUDGE_FILE = f'{EVAL_DIR}/judge_cache.json'
JUDGE = json.load(open(JUDGE_FILE, encoding='utf-8')) if os.path.exists(JUDGE_FILE) else {}

JUDGE_PROMPT = '''질문과 문서 조각 여러 개가 있어요. 각 조각이 이 질문에 답하는 데 **직접 도움이 되는 내용**을 담고 있는지 판정해 주세요.
- 질문의 핵심에 대한 정보가 조각 안에 있으면 true
- 주제만 비슷하고 답이 될 내용이 없으면 false

질문: {q}

{chunks}

JSON으로만 답하세요: {{"relevant": [true, false, ...]}}  (조각 개수와 같은 길이)'''

def judge(q, idxs):
    key = q + '|' + ','.join(map(str, idxs))
    if key in JUDGE:
        return JUDGE[key]
    chunks = '\n'.join(f"[{n}] ({DOCS[i].metadata['doc_title']} > {DOCS[i].metadata['section']}) {DOCS[i].metadata['text'][:500]}"
                       for n, i in enumerate(idxs, 1))
    for _ in range(3):
        try:
            resp = client.chat.completions.create(
                model=CHAT_MODEL, temperature=0, response_format={'type': 'json_object'},
                messages=[{'role': 'user', 'content': JUDGE_PROMPT.format(q=q, chunks=chunks)}])
            rel = [bool(x) for x in json.loads(resp.choices[0].message.content)['relevant']][:len(idxs)]
            rel += [False] * (len(idxs) - len(rel))
            JUDGE[key] = rel
            return rel
        except Exception:
            time.sleep(2)
    return [False] * len(idxs)

def judged_summary(detail):
    rows = detail.to_dict('records')
    with ThreadPoolExecutor(8) as ex:
        rels = list(ex.map(lambda r: judge(r['질문'], list(r['top4'])), rows))
    json.dump(JUDGE, open(JUDGE_FILE, 'w', encoding='utf-8'), ensure_ascii=False)
    df = pd.DataFrame([{'방식': r['방식'], 'qid': r['qid'], 'rel': rel} for r, rel in zip(rows, rels)])
    def stat(g):
        first = g['rel'].map(lambda x: next((p for p, v in enumerate(x, 1) if v), None))
        return pd.Series({
            'LLM판정 Hit@4': g['rel'].map(any).mean(),
            'Precision@4': g['rel'].map(lambda x: sum(x) / max(len(x), 1)).mean(),
            'LLM판정 MRR@4': first.map(lambda p: 1 / p if pd.notna(p) else 0).mean(),
        })
    return df.groupby('방식', sort=False).apply(stat).round(3)

JUDGED = judged_summary(DETAIL)
JUDGED

## 8. 결과 저장

In [ ]:
stamp = time.strftime('%m%d_%H%M')
SUMMARY.to_csv(f'{EVAL_DIR}/summary_{stamp}.csv', encoding='utf-8-sig')
if 'JUDGED' in globals():
    JUDGED.to_csv(f'{EVAL_DIR}/judged_{stamp}.csv', encoding='utf-8-sig')
DETAIL.drop(columns=['top4']).to_csv(f'{EVAL_DIR}/detail_{stamp}.csv', index=False, encoding='utf-8-sig')
print('저장:', f'{EVAL_DIR}/summary_{stamp}.csv')
SUMMARY